# DFT energy-offset refit (refloxide)

Same workflow as `fit_dft_fix.ipynb`, but one shared `DispersiveStructure` / `DispersiveReflectModel` and a single `BatchedGlobalObjective`.

Requires **refloxide** on the kernel (e.g. `make develop` in the sibling `refloxide` repo). Data via `utils.read_fit` / `read_ooc`.

In [ ]:
from refloxide.pxr.plugin.batched_global import BatchedFitter
from refloxide.pxr.plugin.dispersive_instrument import safely_setp_param
from refloxide.pxr.plugin.dispersive_model import select
from refloxide.pxr.plugin.dft_fit import (
    DIAGNOSTIC_ENERGY_EV,
    apply_dft_diagnostic_structure_constraints,
    batched_objective_from_fit_bundle,
    build_dft_dispersive_model,
    link_instrument_to_diagnostic,
    tighten_theta_offset_bounds_from_terms,
)
from refloxide.pxr.plugin.fitters import LogpExtra
from refloxide.pxr.energy import OocAnchor
from utils import read_fit, read_ooc

In [ ]:
dft_constrained = read_fit("dft/dft_en_offset_new2.pkl", material="znpc")
ooc_df = read_ooc("dft.csv", material="znpc")
oocs = OocAnchor.from_dataframe(ooc_df)
len(dft_constrained.objectives)

In [ ]:
diag_obj = next(
    o for o in dft_constrained.objectives
    if abs(float(o.model.energy) - DIAGNOSTIC_ENERGY_EV) < 1e-3
)
float(diag_obj.model.energy)

In [ ]:
model = build_dft_dispersive_model(
    dft_constrained,
    oocs,
    apply_constraints=False,
)
model.pol = "sp"
model

In [ ]:
apply_dft_diagnostic_structure_constraints(model, diag_obj)
link_instrument_to_diagnostic(model)
model.energy_offset.at(DIAGNOSTIC_ENERGY_EV).vary

In [ ]:
objective = batched_objective_from_fit_bundle(
    model,
    dft_constrained,
    tighten_theta_bounds=False,
)
tighten_theta_offset_bounds_from_terms(model, objective.terms)
objective.logp_extra = LogpExtra(objective)
len(objective.varying_parameters()), objective.logl()

In [ ]:
select(model, "ZnPc").thick.value, model.instrument_at(285.1).theta_offset_s.bounds

In [ ]:
fitter = BatchedFitter(objective)
fitter.fit(method="L-BFGS-B", verbose=True)